# 03. Те же проверки для всех девяти элементов

Ответ на вопрос «почему только Pb»: ниже одна и та же схема LOWO для каждого элемента. R² ≤ 0 означает, что модель не лучше медианы склада; для таких элементов прогноз не публикуется. Rb и Co имеют сигнал, но без заключения геохимика нельзя считать его признаком загрязнения (возможен природный фон).

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib
matplotlib.rcParams.update({'figure.dpi':110,'axes.grid':True,'grid.alpha':.3})
import matplotlib.pyplot as plt
import mltj
sites, pts, cov = mltj.load()
print(len(sites), 'складов,', len(pts), 'точек')

In [ ]:
rows=[]
for e in mltj.ELEMENTS:
    y=mltj.target(pts,e)
    b=np.mean([mltj.lowo_r2(pts,y,mltj.BASE,s)[0] for s in range(5)])
    w=[mltj.lowo_r2(pts,y,mltj.FEATURES,s)[0] for s in range(5)]
    rows.append((e,b,np.mean(w),np.std(w)))
res=pd.DataFrame(rows,columns=['элемент','R² базовый','R² вода+pH','sd по зёрнам']).set_index('элемент').sort_values('R² вода+pH',ascending=False)
res.round(3)

In [ ]:
fig,ax=plt.subplots(figsize=(6.4,3.4)); col=['#2e7d32' if v>0.1 else ('#f9a825' if v>0 else '#c62828') for v in res['R² вода+pH']]
ax.bar(res.index,res['R² вода+pH'],yerr=res['sd по зёрнам'],color=col); ax.axhline(0,color='k',lw=1); ax.set_ylabel('LOWO R²'); ax.set_title('Переносимый сигнал по элементам (вода + pH)'); plt.show()

## Доля дисперсии внутри склада

Если почти вся изменчивость элемента приходится на различия между складами, модель с нормированием на склад работает с остаточной частью. Доля внутри склада по логарифму:

In [ ]:
share={}
for e in mltj.ELEMENTS:
    l=np.log(pts[f'{e}_mgkg']); share[e]=np.var(mltj.target(pts,e))/np.var(l)
pd.Series(share,name='доля внутри склада').round(2).to_frame().T

## Что даёт расчёт по всем элементам

- Показывает, для каких элементов модель вообще применима (здесь: Rb, Co, Pb).
- Для остальных шести публикуется только таблица качества, не карта.
- Новые склады и пробы позволят повторить эту же таблицу и увидеть, у каких элементов появляется переносимый сигнал.